In [ ]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community langchain-text-splitters networkx pypdf faiss-cpu fastapi uvicorn tiktoken
!pip install langchain==0.1.17
!pip install langchain-openai==0.0.6
!pip install langchain-core==0.1.53 
!pip install langchain-community==0.0.33
!pip install networkx
!pip install pypdf
!pip install faiss-cpu
!pip install fastapi uvicorn tiktoken

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

In [ ]:
# --- STEP 1: LOAD ---
loader = PyPDFLoader("HR_Policy_Manual_2023.pdf")
data = loader.load()

In [ ]:
# --- STEP 2: CHUNK ---
# We use overlap to ensure context isn't lost at the cut-off points
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

In [ ]:
# --- STEP 3: STORE (Vector DB) ---
# This turns text into numbers (embeddings) and saves them locally
embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(chunks, embeddings)

In [ ]:
# --- STEP 4: RETRIEVE & CHAT ---
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # "Stuffs" all retrieved chunks into the prompt
    retriever=vector_db.as_retriever()
)

In [ ]:
# Example Query
query = "How many leave employee can take?"
response = qa_chain.invoke(query)
print(response["result"])